# The Sigma-Point (Unscented) Kalman Filter

*Course 3 — Nonlinear Kalman Filters, Part 2. The sigma-point KF (SPKF), whose best-known variant is the **Unscented Kalman Filter (UKF)**, removes the EKF's Jacobians by approximating the **distribution** instead of the **function**. Same [six SPI steps](07_Sequential_Probabilistic_Inference_Six_Steps.ipynb).*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 The Key Idea — Approximate the Distribution, Not the Function

- Julier & Uhlmann's insight: *"it is easier to approximate a probability distribution than an arbitrary nonlinear function."*

  → Rather than linearize $f$ (EKF), choose a small set of **deterministic sample points** that capture the input distribution's mean and covariance, push each **through the exact nonlinearity**, and recompute the mean and covariance of the transformed points. This is the **unscented transform**.

- **→ Intuition:** the EKF asks "what's the tangent plane here?"; the SPKF asks "if I place a few well-chosen probes on the input Gaussian and watch where they land, what Gaussian do they trace out?" No derivatives — only function evaluations.

### 🧩 Sigma Points and Weights

- For an $L$-dimensional random vector with mean $\bar{x}$ and covariance $\Sigma_{\tilde{x}}$, build $2L+1$ **sigma points**:

$$
\mathcal{X}^{0} = \bar{x}, \qquad \mathcal{X}^{i} = \bar{x} + \gamma\,\big(\sqrt{\Sigma_{\tilde{x}}}\big)_i, \qquad \mathcal{X}^{i+L} = \bar{x} - \gamma\,\big(\sqrt{\Sigma_{\tilde{x}}}\big)_i,
$$

  where $(\sqrt{\Sigma_{\tilde x}})_i$ is the $i$-th column of a matrix square root (Cholesky) and $\gamma$ is a spread parameter.

  → Put one point at the mean and a symmetric pair along each covariance "axis," scaled by $\gamma$. Together they **exactly reproduce** the mean and covariance of the input Gaussian.

- Transformed statistics use fixed **weights** $\alpha_j^{(m)}$ (mean) and $\alpha_j^{(c)}$ (covariance):

$$
\bar{y} = \sum_{j=0}^{2L}\alpha_j^{(m)}\mathcal{Y}^{j}, \qquad
\Sigma_{\tilde{y}} = \sum_{j=0}^{2L}\alpha_j^{(c)}\big(\mathcal{Y}^{j}-\bar{y}\big)\big(\mathcal{Y}^{j}-\bar{y}\big)^{T}, \qquad \mathcal{Y}^{j}=f(\mathcal{X}^{j}).
$$

  → Pass every sigma point through the true nonlinearity, then take **weighted** averages for the new mean and covariance. Different weight/spread choices give the **UKF** ($\alpha,\beta,\kappa$) or the **central-difference KF** ($h$); UKF's $\beta$ injects known prior knowledge (2 for Gaussians).

- **→ Intuition:** the transform is **accurate to 2nd order** (3rd for symmetric distributions) — strictly better than the EKF's 1st order — for the *same* handful of function evaluations.

### 🧩 The Augmented State

- When noise enters nonlinearly, augment the state with the process and sensor noise so the sigma points also probe the noise dimensions:

$$
x_k^{a} = \begin{bmatrix} x_k \\ w_k \\ v_k\end{bmatrix}, \qquad
\Sigma_{\tilde{x}}^{a} = \begin{bmatrix} \Sigma_{\tilde{x}} & 0 & 0 \\ 0 & \Sigma_{\tilde{w}} & 0 \\ 0 & 0 & \Sigma_{\tilde{v}}\end{bmatrix}.
$$

  → Stack state + noises into one augmented vector with a block-diagonal covariance. Now the unscented transform naturally captures how the **noise's** uncertainty flows through the nonlinear $f$ and $h$ — not just the state's.

- **→ Intuition:** this is the SPKF's counterpart to the EKF's $\hat B\Sigma_{\tilde w}\hat B^T$ and $\hat D\Sigma_{\tilde v}\hat D^T$ terms — but exact to 2nd order and derivative-free. (If noise is purely *additive*, a cheaper "non-augmented" SPKF just adds $\Sigma_{\tilde w},\Sigma_{\tilde v}$ directly.)

### 🧩 The Six SPKF Steps

Using augmented sigma points $\mathcal{X}_{k-1}^{a} = \{\mathcal X^x, \mathcal X^w, \mathcal X^v\}$:

**Prediction**
$$
\textbf{1a: } \mathcal{X}_{k}^{x,i} = f(\mathcal{X}_{k-1}^{x,i}, u_{k-1}, \mathcal{X}_{k-1}^{w,i}), \quad \hat{x}_k^- = \sum_i \alpha_i^{(m)}\mathcal{X}_{k}^{x,i}
$$
$$
\textbf{1b: } \Sigma_{\tilde{x},k}^- = \sum_i \alpha_i^{(c)}\big(\mathcal{X}_{k}^{x,i}-\hat{x}_k^-\big)\big(\mathcal{X}_{k}^{x,i}-\hat{x}_k^-\big)^T
$$
$$
\textbf{1c: } \mathcal{Z}_k^{i} = h(\mathcal{X}_{k}^{x,i}, u_k, \mathcal{X}_{k-1}^{v,i}), \quad \hat{z}_k = \sum_i \alpha_i^{(m)}\mathcal{Z}_k^{i}
$$

**Correction**
$$
\textbf{2a: } \Sigma_{\tilde{z},k} = \sum_i \alpha_i^{(c)}(\mathcal{Z}_k^i-\hat{z}_k)(\mathcal{Z}_k^i-\hat{z}_k)^T, \quad
\Sigma_{\tilde{x}\tilde{z},k} = \sum_i \alpha_i^{(c)}(\mathcal{X}_k^{x,i}-\hat{x}_k^-)(\mathcal{Z}_k^i-\hat{z}_k)^T, \quad
L_k = \Sigma_{\tilde{x}\tilde{z},k}\Sigma_{\tilde{z},k}^{-1}
$$
$$
\textbf{2b: } \hat{x}_k^+ = \hat{x}_k^- + L_k(z_k - \hat{z}_k) \qquad
\textbf{2c: } \Sigma_{\tilde{x},k}^+ = \Sigma_{\tilde{x},k}^- - L_k\Sigma_{\tilde{z},k}L_k^T
$$

  → Identical *shape* to the linear/EK filter, but every mean and covariance is built by **passing sigma points through the true $f,h$ and taking weighted sums** — no Jacobians anywhere. The gain still equals cross-covariance ÷ innovation-covariance.

- **→ Intuition:** the SPKF plugs into the same six-step template; only *how* the statistics are computed changed (sample-and-reweight instead of differentiate).

### 🧩 SPKF vs. EKF

- **No Jacobians:** the SPKF needs only function evaluations of $f,h$ — no analytic derivatives to derive or code. Huge practical win for complex models.

- **Higher accuracy:** captures mean/covariance to **2nd order** (vs. EKF's 1st), so it handles stronger nonlinearity and is far less prone to divergence.

- **Comparable cost:** $2L+1$ evaluations of $f,h$ per step ≈ the cost of forming EKF Jacobians; both are $\mathcal{O}(L^3)$ overall.

- **Derivative-free & modular:** swap in any $f,h$ (even table-lookups or black boxes) without re-deriving math.

  → The SPKF is usually the **default choice** for nonlinear Gaussian filtering unless the model is nearly linear (where the EKF is fine) or highly non-Gaussian/multimodal (where a **particle filter**, Course 4, is needed).

- **→ Intuition:** same computational ballpark as the EKF, but more accurate and vastly easier to implement correctly — you never differentiate anything.

### 🧩 Summary

- The **SPKF/UKF** approximates the **distribution** with a set of deterministic **sigma points** and pushes them through the *exact* nonlinearity (the **unscented transform**) — no linearization, no Jacobians.

- Sigma points reproduce the input mean and covariance; weighted sums of the transformed points give the output mean, covariance, and cross-covariance — accurate to **2nd order**.

- The **augmented state** (state + process + sensor noise) lets the transform capture nonlinearly-entering noise; additive noise allows a cheaper variant.

- It fills the **same six steps** as every Kalman filter, and generally beats the EKF on accuracy and ease of implementation at similar cost.

---
*Next: [14 · Parameter Estimation — Joint & Dual Filters](14_Parameter_Estimation_Joint_Dual.ipynb).*